# Detailed Notebook 3: Solving ODEs with `solve_ivp` + Integration Techniques

**What is a differential equation and why solve it numerically?**

A differential equation relates a quantity to its own rate of change.  Virtually
every equation of motion in classical and modern physics is an ODE:

* Newton's second law: m·d²x/dt² = F(x, t)
* Radioactive decay: dN/dt = −λN
* LRC circuit: L·d²q/dt² + R·dq/dt + q/C = V(t)
* General relativity (geodesics), Schrödinger equation, fluid dynamics …

Most ODEs have **no closed-form analytic solution**.  We solve them numerically:
step forward in time, approximating the solution at each small time step.

**Contents**
1. The Initial Value Problem and the state-vector trick
2. `solve_ivp` — function signature and key arguments
3. Example: radioactive decay (1st-order ODE)
4. Example: pendulum (2nd-order → 1st-order system)
5. Example: driven damped oscillator
6. Example: orbits (2-D system)
7. Event detection — stopping at a condition
8. Choosing a solver and tolerances
9. Numerical integration (`quad`, `trapezoid`)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp, quad

---
## Section 1: The Initial Value Problem and the state-vector trick

### What is an IVP?

An *Initial Value Problem* (IVP) consists of:
1. An ODE: dy/dt = f(t, y)
2. An initial condition: y(t_start) = y_0

`solve_ivp` advances the solution from t_start to t_end starting from y_0.

### Converting a higher-order ODE to a first-order system

`solve_ivp` only accepts **first-order** ODEs: dy/dt = f(t, y).
A 2nd-order equation like Newton's law m·ẍ = F(x, ẋ, t) must be
rewritten using the *state vector* trick:

    Let  u = [x, v]  where v = dx/dt

Then:
    du/dt = [dx/dt, dv/dt] = [v, F(x,v,t)/m]

This is a first-order system in the 2-component vector u.

**General rule:** an N-th order ODE becomes N coupled first-order ODEs.

---
## Section 2: `solve_ivp` — function signature and key arguments

```python
sol = solve_ivp(
    fun,          # rhs function: fun(t, y) → dy/dt  (array of same shape as y)
    t_span,       # (t_start, t_end)
    y0,           # initial state vector (list or array)
    method='RK45',# solver algorithm (see Section 8)
    t_eval=...,   # array of times at which to record the solution
    rtol=1e-3,    # relative tolerance (controls accuracy)
    atol=1e-6,    # absolute tolerance
    events=...,   # optional event functions (see Section 7)
)
```

### What `solve_ivp` returns

| `sol.t` | Array of times the solution was evaluated at |
|---------|----------------------------------------------|
| `sol.y` | 2-D array; `sol.y[i, :]` is the i-th state variable over time |
| `sol.success` | True if solver reached `t_end` without error |
| `sol.message` | Explains why the solver stopped |

**Accessing the solution:**
```python
sol.y[0, :]   # first state variable at all times  → x(t)
sol.y[1, :]   # second state variable at all times → v(t)
```

---
## Section 3: Radioactive decay (1st-order ODE)

### The physics

dN/dt = −λ·N

The decay rate is proportional to the number of atoms present.  The
analytic solution is N(t) = N₀·exp(−λt).

### Translating to code

The right-hand side function (rhs) must take `(t, y)` and return `dy/dt`:

* `t` is the current time (we don't use it here because the equation is autonomous)
* `y` is the current state — here just `[N]`
* The function returns `[-lambda * N]`

In [ ]:
# ── Example 3: radioactive decay ────────────────────────────────────────────
decay_constant = 0.8   # s^-1
N0 = 1000.0            # initial number of atoms

# rhs: right-hand side function.
# y is a 1-element array [N]; we return [dN/dt]
def decay_rhs(t, y):
    N = y[0]
    dNdt = -decay_constant * N
    return [dNdt]

# Solve from t=0 to t=8 seconds
t_start, t_end = 0.0, 8.0
t_eval = np.linspace(t_start, t_end, 300)

sol = solve_ivp(decay_rhs,
                t_span=[t_start, t_end],
                y0=[N0],
                t_eval=t_eval)

# sol.y has shape (1, 300) — 1 state variable, 300 time points
N_numerical = sol.y[0, :]        # extract the N(t) array
N_analytic  = N0 * np.exp(-decay_constant * t_eval)

plt.figure(figsize=(7, 4))
plt.plot(sol.t, N_numerical, label='solve_ivp (RK45)', linewidth=2)
plt.plot(t_eval, N_analytic, 'k--', label='analytic N₀ e^(−λt)', linewidth=1.5)
plt.xlabel('t (s)')
plt.ylabel('N(t)')
plt.title('Radioactive decay: numerical vs analytic')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Check the maximum error
max_err = np.max(np.abs(N_numerical - N_analytic))
print(f'Maximum error vs analytic: {max_err:.6e}  (with default tolerances)')

**Step-by-step walkthrough:**

1. `decay_rhs(t, y)` is the function that tells `solve_ivp` how fast the
   state is changing at every moment.  It takes the *current* state `y=[N]`
   and returns the *rate of change* `[dN/dt]`.
2. `solve_ivp` starts at `y0=[1000]` and applies the RK45 algorithm
   (4th-order Runge-Kutta) in small steps, building up the solution.
3. `t_eval` tells it which times to record — the solver may take intermediate
   steps for accuracy but will always record at these times.
4. The result in `sol.y[0, :]` should match the analytic exponential very closely.

---
## Section 4: Pendulum (2nd-order ODE → 2-component state vector)

### The physics

Equation of motion for a simple pendulum:

    d²θ/dt² = −(g/L) · sin(θ)

This is a 2nd-order ODE.  We convert it using the state vector:

    u = [θ, ω]  where  ω = dθ/dt

    du/dt = [dθ/dt, dω/dt] = [ω, −(g/L)·sin(θ)]

In [ ]:
# ── Example 4: pendulum motion ───────────────────────────────────────────────
g, L = 9.81, 1.0   # m/s², m

def pendulum_rhs(t, u):
    theta, omega = u[0], u[1]
    dtheta_dt = omega
    domega_dt = -(g/L) * np.sin(theta)
    return [dtheta_dt, domega_dt]

# Initial conditions: theta_0 = 0.7 rad, omega_0 = 0 (released from rest)
u0 = [0.7, 0.0]

t_eval = np.linspace(0, 12, 1200)
sol_pend = solve_ivp(pendulum_rhs,
                     t_span=[0, 12],
                     y0=u0,
                     t_eval=t_eval,
                     rtol=1e-8, atol=1e-10)

theta = sol_pend.y[0, :]
omega = sol_pend.y[1, :]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(sol_pend.t, np.degrees(theta))
axes[0].set_xlabel('t (s)');  axes[0].set_ylabel('θ (degrees)')
axes[0].set_title('Pendulum angle vs time');  axes[0].grid(True, alpha=0.3)

# Phase space plot: theta vs omega — a closed loop for an energy-conserving system
axes[1].plot(np.degrees(theta), np.degrees(omega))
axes[1].set_xlabel('θ (degrees)');  axes[1].set_ylabel('ω (degrees/s)')
axes[1].set_title('Phase space (θ vs ω)');  axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**What to notice:**

- **Left plot:** The pendulum oscillates in angle — not perfectly sinusoidal
  because the amplitude (0.7 rad ≈ 40°) is not small.
- **Right plot (phase space):** θ vs ω forms a closed ellipse-like loop.
  Closure confirms energy conservation — the solver is accurate.

**Tightening tolerances** (`rtol=1e-8, atol=1e-10`) makes the phase-space
loop close more precisely.  For long simulations where energy drift matters,
always tighten tolerances.

---
## Section 5: Driven damped oscillator (forcing + damping)

### The physics

    m·ẍ + b·ẋ + k·x = F₀·cos(ω_d·t)

In canonical form (m=1):

    ẍ = −2ζω₀·ẋ − ω₀²·x + (F₀/m)·cos(ω_d·t)

State vector: u = [x, ẋ]

In [ ]:
# ── Example 5: driven damped oscillator ─────────────────────────────────────
omega0 = 2 * np.pi * 1.0   # natural frequency: 1 Hz
zeta   = 0.08               # damping ratio (underdamped)
F0     = 1.0                # driving force amplitude
omega_d = 2 * np.pi * 0.9  # driving frequency (near resonance)

def driven_osc_rhs(t, u):
    x, xdot = u[0], u[1]
    xddot = (-2*zeta*omega0*xdot
             - omega0**2 * x
             + F0 * np.cos(omega_d * t))
    return [xdot, xddot]

t_eval = np.linspace(0, 30, 3000)
sol_osc = solve_ivp(driven_osc_rhs,
                    t_span=[0, 30],
                    y0=[0.0, 0.0],
                    t_eval=t_eval,
                    rtol=1e-8, atol=1e-10)

plt.figure(figsize=(10, 4))
plt.plot(sol_osc.t, sol_osc.y[0, :])
plt.xlabel('t (s)')
plt.ylabel('x(t)')
plt.title(f'Driven damped oscillator  ω₀={omega0/(2*np.pi):.1f} Hz, '
          f'ω_d={omega_d/(2*np.pi):.1f} Hz, ζ={zeta}')
plt.grid(True, alpha=0.3)
plt.show()

**What to notice:**
- The early part of the curve shows the *transient* — the system is adjusting
  from x=0, ẋ=0 to the driving frequency.
- After several cycles the *steady state* is reached: constant-amplitude oscillation.
- Because ω_d is close to ω₀ and damping is light, the amplitude is large (resonance!).
  Try changing `omega_d` to `2*pi*0.5` (far from resonance) and compare.

---
## Section 6: Planetary orbit in 2-D (4-component state vector)

### The physics

Gravity gives radial acceleration toward the Sun:

    a = −GM/r²  ·  r̂

In Cartesian coordinates:

    ẍ = −GM · x / r³
    ÿ = −GM · y / r³

State vector: u = [x, y, vx, vy]  (4 components)

In [ ]:
# ── Example 6: elliptical orbit ─────────────────────────────────────────────
from scipy.constants import G

G_SI  = G
M_sun = 2e30      # kg
m_earth = 6e24    # kg (not needed for trajectory since it cancels)
AU = 1.496e11     # metres per AU

# Scale to AU and years for nicer numbers
# GM_sun in AU^3/yr^2
GM_year = G_SI * M_sun * (3.156e7)**2 / AU**3   # ≈ 4π² AU³/yr²

def orbit_rhs(t, u):
    x, y, vx, vy = u
    r3 = (x**2 + y**2)**1.5
    ax = -GM_year * x / r3
    ay = -GM_year * y / r3
    return [vx, vy, ax, ay]

# Initial conditions: Earth at perihelion (closest approach)
# Perihelion distance ≈ 0.983 AU, velocity ≈ 2π AU/yr * (1+e) ≈ 6.28*1.017 AU/yr
r_peri = 0.983   # AU
v_peri = 2*np.pi * 1.017  # AU/yr (circular speed * correction for eccentricity)

u0_orbit = [r_peri, 0.0, 0.0, v_peri]

t_eval = np.linspace(0, 1.0, 1000)   # one year
sol_orbit = solve_ivp(orbit_rhs,
                      t_span=[0, 1.0],
                      y0=u0_orbit,
                      t_eval=t_eval,
                      rtol=1e-9, atol=1e-11)

x = sol_orbit.y[0, :]
y = sol_orbit.y[1, :]
r = np.sqrt(x**2 + y**2)

plt.figure(figsize=(7, 7))
plt.plot(x, y, label='Earth orbit')
plt.scatter([0], [0], color='orange', s=200, zorder=5, label='Sun')
plt.xlabel('x (AU)');  plt.ylabel('y (AU)')
plt.title('Earth orbit (1 year)')
plt.axis('equal')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f'Min distance from Sun: {r.min():.4f} AU  (perihelion)')
print(f'Max distance from Sun: {r.max():.4f} AU  (aphelion)')

---
## Section 7: Event detection — stopping at a condition

Sometimes we want the solver to **stop** when a condition is met, rather than
running to a fixed end time.

Examples:
- A ball hits the ground (y = 0)
- A pendulum passes through vertical (θ = 0)
- A satellite reaches a specific altitude

### How events work

An event function `e(t, y)` returns a scalar.  `solve_ivp` monitors its sign
and triggers when it **crosses zero**.

| Attribute | Meaning |
|-----------|---------|
| `e.terminal = True` | Stop the solver when the event is triggered |
| `e.terminal = False` | Record the event time but keep going |
| `e.direction = -1` | Only trigger on a zero-crossing going negative |
| `e.direction = +1` | Only trigger on a zero-crossing going positive |
| `e.direction = 0` (default) | Trigger on any zero-crossing |

In [ ]:
# ── Example 7a: projectile — find impact time and range ─────────────────────
g_acc = 9.81   # m/s²
theta0 = np.radians(35)  # launch angle
v0     = 25.0            # launch speed m/s

def projectile_rhs(t, u):
    x, y, vx, vy = u
    return [vx, vy, 0.0, -g_acc]

# Event: ball hits ground (y = 0, going downward)
def hits_ground(t, u):
    return u[1]       # y component of state
hits_ground.terminal  = True
hits_ground.direction = -1   # only trigger when y is decreasing (falling)

u0_proj = [0.0, 0.0, v0*np.cos(theta0), v0*np.sin(theta0)]

sol_proj = solve_ivp(projectile_rhs,
                     t_span=[0, 10],
                     y0=u0_proj,
                     events=hits_ground,
                     max_step=0.01)

t_impact = sol_proj.t_events[0][0]   # time when event occurred
state_at_impact = sol_proj.y_events[0][0]

print(f'Impact time:  {t_impact:.4f} s')
print(f'Range (x at impact): {state_at_impact[0]:.4f} m')

# Analytic range for comparison
R_analytic = v0**2 * np.sin(2*theta0) / g_acc
print(f'Analytic range: {R_analytic:.4f} m')

plt.figure(figsize=(8, 4))
plt.plot(sol_proj.y[0, :], sol_proj.y[1, :], label='trajectory')
plt.scatter([state_at_impact[0]], [0], color='red', zorder=5, s=80, label='impact')
plt.xlabel('x (m)');  plt.ylabel('y (m)')
plt.title(f'Projectile: θ={np.degrees(theta0):.0f}°, v₀={v0} m/s')
plt.legend();  plt.grid(True, alpha=0.3);  plt.show()

In [ ]:
# ── Example 7b: pendulum — count zero crossings ──────────────────────────────
# Count how many times the pendulum passes through θ=0 going left→right
# (i.e. direction=+1) in 20 seconds.

def passes_vertical(t, u):
    return u[0]   # theta = 0 means vertical
passes_vertical.terminal  = False   # don't stop, just record
passes_vertical.direction = +1      # only when theta is increasing (swinging right)

g, L = 9.81, 0.5   # shorter pendulum
def pend_rhs2(t, u):
    return [u[1], -(g/L)*np.sin(u[0])]

sol_pend2 = solve_ivp(pend_rhs2,
                      t_span=[0, 20],
                      y0=[0.5, 0.0],
                      events=passes_vertical,
                      rtol=1e-8, atol=1e-10,
                      dense_output=True)

crossing_times = sol_pend2.t_events[0]
print(f'Number of θ=0 upswing crossings in 20 s: {len(crossing_times)}')
print(f'Period estimate from crossings: {np.diff(crossing_times).mean():.4f} s')
print(f'Small-angle period: {2*np.pi*np.sqrt(L/g):.4f} s')

---
## Section 8: Choosing a solver and tolerances

### Available methods in `solve_ivp`

| Method | Type | When to use |
|--------|------|-------------|
| `'RK45'` | Explicit, adaptive (default) | Most non-stiff problems |
| `'RK23'` | Explicit, 2nd/3rd order | Quick check, low accuracy OK |
| `'DOP853'`| Explicit, 8th order | High accuracy needed |
| `'Radau'` | Implicit | **Stiff** systems (fast + slow timescales) |
| `'BDF'` | Implicit | **Stiff**, large systems |
| `'LSODA'` | Auto-switching | When unsure if stiff |

### What is a *stiff* system?

Stiffness means the ODE contains both very fast and very slow dynamics.
Example: a circuit with resistors spanning many orders of magnitude.

With explicit methods (RK45) the step size is limited by the fastest dynamics,
making the solver take millions of tiny steps just to resolve the slow dynamics.
Implicit methods (Radau/BDF) handle this efficiently.

### Tolerance guidance

```python
# Default (fast, moderate accuracy)
rtol=1e-3, atol=1e-6

# Physics-grade (recommended for pendulum, orbit, etc.)
rtol=1e-8, atol=1e-10

# Very high precision (long-time energy conservation)
rtol=1e-12, atol=1e-14
```

In [ ]:
# ── Example 8: effect of tolerance on energy conservation ───────────────────
g, L = 9.81, 1.0

def pend_rhs3(t, u):
    return [u[1], -(g/L)*np.sin(u[0])]

def pendulum_energy(theta, omega, L=1.0):
    # E = (1/2)*L^2*omega^2 - g*L*cos(theta)   (per unit mass)
    return 0.5 * L**2 * omega**2 - g * L * np.cos(theta)

u0 = [1.0, 0.0]   # large amplitude: 1 radian
t_eval = np.linspace(0, 100, 5000)

for tol_label, rtol, atol in [('loose (1e-3)', 1e-3, 1e-6),
                               ('tight (1e-9)', 1e-9, 1e-11)]:
    s = solve_ivp(pend_rhs3, [0, 100], u0, t_eval=t_eval, rtol=rtol, atol=atol)
    E = pendulum_energy(s.y[0, :], s.y[1, :])
    drift = (E.max() - E.min()) / abs(E.mean())
    print(f'{tol_label:18s}:  relative energy drift = {drift:.2e}')

---
## Section 9: Numerical integration with `quad` and `trapezoid`

Even without an ODE, physics often requires integrating a function that has no
closed-form antiderivative, or that is only known at discrete points.

### Two strategies

| Situation | Tool |
|-----------|------|
| Function is known analytically (or via a lambda) | `scipy.integrate.quad` |
| Function is known only at discrete sample points | `np.trapezoid` |

### `quad` — adaptive quadrature

`quad` uses an adaptive Gauss-Kronrod rule.  It subdivides the interval where
the integrand is rapidly changing to achieve the requested tolerance.

```python
I, err = quad(f, a, b)
# I   = numerical integral value
# err = estimated absolute error
```

### `np.trapezoid` — trapezoidal rule on discrete data

Approximates the area under the curve by connecting data points with
straight lines (trapezoids) and summing their areas.

```python
I = np.trapezoid(y_array, x_array)
```

In [ ]:
# ── Example 9a: quad on a smooth function ────────────────────────────────────
from scipy.integrate import quad

# Integral of  e^(-x^2)  from 0 to infinity = sqrt(pi)/2
I_gauss, err_gauss = quad(lambda x: np.exp(-x**2), 0, np.inf)
print(f'quad: ∫₀^∞ e^(-x²) dx = {I_gauss:.10f}')
print(f'exact (sqrt(pi)/2)     = {np.sqrt(np.pi)/2:.10f}')
print(f'estimated error:         {err_gauss:.2e}')

# Integral of sin(x)/x from 0 to pi (the sinc integral)
# Note: integrand has a removable singularity at x=0; quad handles this
def sinc_integrand(x):
    return np.sinc(x / np.pi)   # numpy.sinc is sin(pi*x)/(pi*x)
I_sinc, err_sinc = quad(sinc_integrand, 0, np.pi)
print(f'\nquad: ∫₀^π sin(x)/x dx = {I_sinc:.10f}')

In [ ]:
# ── Example 9b: trapezoid on discrete data ───────────────────────────────────
# Simulate an accelerometer recording (acceleration vs time)
# and compute the velocity change (impulse/mass = Δv).

rng = np.random.default_rng(5)
t_acc = np.linspace(0, 2.0, 50)
# Acceleration pulse: Gaussian shape + noise
a_acc = 4.0 * np.exp(-((t_acc - 1.0)**2) / 0.1) + rng.normal(0, 0.1, len(t_acc))

# Trapezoidal integral = area under a(t) = Δv
delta_v = np.trapezoid(a_acc, t_acc)
print(f'Measured Δv from accelerometer = {delta_v:.4f} m/s')

# Exact Δv (without noise, from the Gaussian pulse)
from scipy.integrate import quad as scipy_quad
delta_v_exact, _ = scipy_quad(lambda t: 4.0*np.exp(-((t-1.0)**2)/0.1), 0, 2)
print(f'Expected Δv (no noise)          = {delta_v_exact:.4f} m/s')

plt.figure(figsize=(7, 4))
plt.plot(t_acc, a_acc, 'o-', markersize=4, label='acceleration data')
plt.fill_between(t_acc, a_acc, alpha=0.3, label=f'area = Δv = {delta_v:.2f} m/s')
plt.xlabel('t (s)');  plt.ylabel('a (m/s²)')
plt.title('np.trapezoid: area under acceleration = velocity change')
plt.legend();  plt.grid(True, alpha=0.3);  plt.show()

---
## Summary

```
ODE system:
  1. Write the physics as du/dt = f(t, u)
  2. Convert N-th order ODE to N coupled 1st-order ODEs via state vector
  3. Call solve_ivp(f, [t0, tf], u0, t_eval=..., rtol=..., atol=...)
  4. Access solution: sol.y[0, :] = first state variable, etc.
  5. Check sol.success and plot residuals / energy conservation

Integration:
  • Known function:   I, err = quad(f, a, b)
  • Discrete data:    I = np.trapezoid(y, x)
```